In [235]:
import zipfile
import numpy as np
import pandas as pd

In [236]:
# Read in data
HMRpath = "../data/raw/UGPR91DT.zip"

with zipfile.ZipFile(HMRpath) as zip_ref:
    zip_ref.extractall("../data/raw")
    print("Extracted files:")
    for file in zip_ref.namelist():
        print(file)

rawHMR= pd.read_stata("../data/raw/UGPR91FL.dta")

Extracted files:
UGPR91FL.frq
UGPR91FL.frw
UGPR91FL.map
UGPR91FL.dta
UGPR91FL.DO
UGPR91FL.DCT


In [ ]:
# TEST
sh = rawHMR[rawHMR["hhid"] == "       1   1"]
cols = sh[["hvidx","hv005", "hv024", "hv103", "hml1", "hml1a", 'hml2', "hml10", "hmlidx", "hml3", "hml4", "hml5", "hml6", "hml7", "hml8", "hml11", "hml12", "hmla", "hmlb", "hmlc", "hmld", "hmle"]]

cols

,hvidx,hv005,hv024,hv103,hml1,hml1a,hml2,hml10,hmlidx,hml3,hml4,hml5,hml6,hml7,hml8,hml11,hml12,hmla,hmlb,hmlc,hmld,hmle
0,1,416820,kampala,yes,2,2,0,"no, don't know",2.0,"yes, seen",0.0,NaN,NaN,other type (not llin),NaN,2.0,only untreated nets,1.0,2.0,NaN,NaN,NaN
1,2,416820,kampala,yes,2,2,0,"no, don't know",2.0,"yes, seen",0.0,NaN,NaN,other type (not llin),NaN,2.0,only untreated nets,1.0,2.0,NaN,NaN,NaN
2,3,416820,kampala,yes,2,2,0,"no, don't know",1.0,"yes, seen",7.0,NaN,NaN,other type (not llin),NaN,4.0,only untreated nets,3.0,4.0,5.0,6.0,NaN
3,4,416820,kampala,yes,2,2,0,"no, don't know",1.0,"yes, seen",7.0,NaN,NaN,other type (not llin),NaN,4.0,only untreated nets,3.0,4.0,5.0,6.0,NaN
4,5,416820,kampala,yes,2,2,0,"no, don't know",1.0,"yes, seen",7.0,NaN,NaN,other type (not llin),NaN,4.0,only untreated nets,3.0,4.0,5.0,6.0,NaN
5,6,416820,kampala,yes,2,2,0,"no, don't know",1.0,"yes, seen",7.0,NaN,NaN,other type (not llin),NaN,4.0,only untreated nets,3.0,4.0,5.0,6.0,NaN


## Variables
**Numerator:** all of de facto household members in region that slept under an ITN last night
**Denominator:** all of de facto household members in the region

`hhid`: Household ID

`hv001`: cluster

`hvidx`: Indivudal line number (individual-level)

`hv005`: Weight (household-level)

`hv024`: Region (household-level)

`hv103`: Stayed last night (y/n/missing) (person-level)

`hmlidx`: Net number (net-level)

`hml1`: Number of nets in household (household-level)

`hml10`: Net is ITN (net-level)

`hml12`: Type of net person slept under last night (person-level)


In [237]:
cols = ['hhid', 'hv001', 'hvidx', 'hv005', 'hv024', 'hv103', 'hmlidx', 
        'hml10', 'hml12', 'hml1', 'hml1a']
hmr = rawHMR[cols]

# Filter to de facto members
hmr_df = hmr[hmr['hv103'] == "yes"].copy()
records_dropped = len(hmr) - len(hmr_df)
print("Records dropped:", records_dropped)

print(hmr_df)

Records dropped: 1352
               hhid  hv001  hvidx  ...                      hml12 hml1 hml1a
0             1   1      1      1  ...        only untreated nets    2     2
1             1   1      1      2  ...        only untreated nets    2     2
2             1   1      1      3  ...        only untreated nets    2     2
3             1   1      1      4  ...        only untreated nets    2     2
4             1   1      1      5  ...        only untreated nets    2     2
...             ...    ...    ...  ...                        ...  ...   ...
49310       342  30    342      2  ...    only treated (itn) nets    2     2
49311       342  30    342      3  ...  did not sleep under a net    2     2
49312       342  30    342      4  ...  did not sleep under a net    2     2
49313       342  30    342      5  ...    only treated (itn) nets    2     2
49314       342  30    342      6  ...    only treated (itn) nets    2     2

[47963 rows x 11 columns]


## ITN Use by Region
The sample design of the 2024-25 Uganda MIS uses separate sampling frames for refugee settlements and non-refugee areas. Therefore the weights are calibrated to make the non-refugee sample representative of the non-refugee Ugandan population. Since the refugee settlements have different weights, 

In [238]:
# Use = weighted sum of "slept under ITN" / weighted sum of all de facto members

# Create "slept under ITN" column
ITN_pos = ['only treated (itn) nets', 'both treated (itn) and untreated nets']
hmr_df['slept_ITN'] = hmr_df['hml12'].isin(ITN_pos)

# Create numerator column (slept_ITN x weight)
hmr_df['use_numerator'] = hmr_df.hv005 * hmr_df.slept_ITN

# Group by region
region_df = hmr_df[['hv024', 'hv005', 'slept_ITN', 'use_numerator']].groupby('hv024').sum().reset_index()

# Drop refugee settlements
region_df = region_df[region_df['hv005'] != 0]

# Divide
region_df['use_rate'] = region_df["use_numerator"]/region_df["hv005"]

region_df

<positron-console-cell-238>:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,hv024,hv005,slept_ITN,use_numerator,use_rate
0,kampala,1014956700,1345,560622900,0.552361
1,south buganda,7169112974,1901,4244450342,0.592047
2,north buganda,5120003785,2104,3361941885,0.656629
3,busoga,3797009509,2042,2461589335,0.648297
4,bukedi,2585607226,2002,1679731306,0.649647
5,elgon,2059554498,2073,1389564440,0.674692
6,teso,2367811911,2384,1845445847,0.779389
7,lango,2437319984,2207,1872548273,0.768282
8,acholi,1953488373,1696,1138711746,0.582912
9,karamoja,1424291664,1630,781591974,0.548758


## Access to ITNs
min(ITNS x 2, household_size) / household_size per household, then weighted mean across all people in the region. 

Not dropping hmlidx NaN values because it's just that a family member isn't linked to a net. 

In [239]:
hmr_df

# Build net-level dataset
nets_df = hmr_df[['hhid', 'hml10', 'hmlidx']]
nets_df = nets_df.dropna(subset='hmlidx').drop_duplicates(subset=['hhid', 'hmlidx'])
itn_counts = (
    nets_df[nets_df['hml10'] == 'yes']
    .groupby('hhid')
    .count()
    .reset_index()
    .rename(columns={'hml10': 'itn_count'})
    .drop(columns='hmlidx')
)

# Merge
hmr_df = hmr_df.merge(itn_counts, how='left', on='hhid')
hmr_df['itn_count'] = hmr_df['itn_count'].fillna(0)
hmr_df

,hhid,hv001,hvidx,hv005,hv024,hv103,hmlidx,hml10,hml12,hml1,hml1a,slept_ITN,use_numerator,itn_count
0,1 1,1,1,416820,kampala,yes,2.0,"no, don't know",only untreated nets,2,2,False,0,0.0
1,1 1,1,2,416820,kampala,yes,2.0,"no, don't know",only untreated nets,2,2,False,0,0.0
2,1 1,1,3,416820,kampala,yes,1.0,"no, don't know",only untreated nets,2,2,False,0,0.0
3,1 1,1,4,416820,kampala,yes,1.0,"no, don't know",only untreated nets,2,2,False,0,0.0
4,1 1,1,5,416820,kampala,yes,1.0,"no, don't know",only untreated nets,2,2,False,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47958,342 30,342,2,0,refugee settlements,yes,2.0,yes,only treated (itn) nets,2,2,True,0,1.0
47959,342 30,342,3,0,refugee settlements,yes,NaN,NaN,did not sleep under a net,2,2,False,0,1.0
47960,342 30,342,4,0,refugee settlements,yes,NaN,NaN,did not sleep under a net,2,2,False,0,1.0
47961,342 30,342,5,0,refugee settlements,yes,2.0,yes,only treated (itn) nets,2,2,True,0,1.0


In [240]:
# ITN capacity
hmr_df['itn_capacity'] = hmr_df['itn_count'] * 2

# Household size
hh_df = hmr_df[['hhid', 'hvidx']]
size = hh_df.groupby('hhid')['hvidx'].nunique().reset_index()
size.columns = ['hhid', 'hh_size']

# Merge
hmr_df = hmr_df.merge(size, how='left', on='hhid')


In [241]:
# Compute ratio
hmr_df['access_ratio'] = np.minimum(hmr_df['itn_capacity'], hmr_df['hh_size']) / hmr_df['hh_size']

hmr_df

,hhid,hv001,hvidx,hv005,hv024,hv103,hmlidx,hml10,hml12,hml1,hml1a,slept_ITN,use_numerator,itn_count,itn_capacity,hh_size,access_ratio
0,1 1,1,1,416820,kampala,yes,2.0,"no, don't know",only untreated nets,2,2,False,0,0.0,0.0,6,0.000000
1,1 1,1,2,416820,kampala,yes,2.0,"no, don't know",only untreated nets,2,2,False,0,0.0,0.0,6,0.000000
2,1 1,1,3,416820,kampala,yes,1.0,"no, don't know",only untreated nets,2,2,False,0,0.0,0.0,6,0.000000
3,1 1,1,4,416820,kampala,yes,1.0,"no, don't know",only untreated nets,2,2,False,0,0.0,0.0,6,0.000000
4,1 1,1,5,416820,kampala,yes,1.0,"no, don't know",only untreated nets,2,2,False,0,0.0,0.0,6,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47958,342 30,342,2,0,refugee settlements,yes,2.0,yes,only treated (itn) nets,2,2,True,0,1.0,2.0,6,0.333333
47959,342 30,342,3,0,refugee settlements,yes,NaN,NaN,did not sleep under a net,2,2,False,0,1.0,2.0,6,0.333333
47960,342 30,342,4,0,refugee settlements,yes,NaN,NaN,did not sleep under a net,2,2,False,0,1.0,2.0,6,0.333333
47961,342 30,342,5,0,refugee settlements,yes,2.0,yes,only treated (itn) nets,2,2,True,0,1.0,2.0,6,0.333333


In [242]:
# calculate access
hmr_df['weighted_ratio'] = hmr_df.hv005 * hmr_df.access_ratio

# Group by region
region_df_access = hmr_df[['hv024', 'hv005', 'weighted_ratio']].groupby('hv024').sum().reset_index()

# Drop refugee settlements
region_df_access = region_df_access[region_df_access['hv005'] != 0]

# Divide
region_df_access['access_rate'] = region_df_access['weighted_ratio']/region_df_access['hv005']

region_df_access

<positron-console-cell-242>:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,hv024,hv005,weighted_ratio,access_rate
0,kampala,1014956700,5.727107e+08,0.564271
1,south buganda,7169112974,4.449315e+09,0.620623
2,north buganda,5120003785,3.370096e+09,0.658221
3,busoga,3797009509,2.394948e+09,0.630746
4,bukedi,2585607226,1.609412e+09,0.622450
5,elgon,2059554498,1.366622e+09,0.663552
6,teso,2367811911,1.832715e+09,0.774012
7,lango,2437319984,1.858230e+09,0.762407
8,acholi,1953488373,1.120168e+09,0.573419
9,karamoja,1424291664,6.527313e+08,0.458285


These don't match because this file doesn't contain information for nets that were owned but not used. 

In [175]:
# trouble shooting
test1 = hmr_df[hmr_df['hhid'] == '       1   4']
test1

# look for households with 3 or 4 nets
#test2 = hmr_df[hmr_df['hml1'] == 3]
#test2

hmr[hmr['hhid'] == '       1   4']

,hhid,hv001,hvidx,hv005,hv024,hv103,hmlidx,hml10,hml12,hml1,hml1a
15,1 4,1,1,416820,kampala,yes,1.0,yes,only treated (itn) nets,3,3
16,1 4,1,2,416820,kampala,yes,2.0,yes,only treated (itn) nets,3,3
17,1 4,1,3,416820,kampala,yes,2.0,yes,only treated (itn) nets,3,3


## Household Recode

In [243]:
# Read in data
HRpath = "../data/raw/UGHR91DT.zip"

with zipfile.ZipFile(HRpath) as zip_ref:
    zip_ref.extractall("../data/raw")
    print("Extracted files:")
    for file in zip_ref.namelist():
        print(file)

rawHR= pd.read_stata("../data/raw/UGHR91FL.dta")

Extracted files:
UGHR91FL.frq
UGHR91FL.frw
UGHR91FL.map
UGHR91FL.dta
UGHR91FL.DO
UGHR91FL.DCT


In [ ]:
#cols = ['hhid', 'hv001', 'hvidx', 'hv005', 'hv024', 'hv103', 'hmlidx', 
        #'hml10', 'hml12', 'hml1', 'hml1a']
#hr = rawHR[cols]

#hr_hml = list(rawHR.filter(like='hml'))
#hr_hml.extend(['hhid', 'hvidx'])


all_cols = ['hhid']
hml_cols = list(rawHR.filter(like='hml'))
all_cols.extend(hml_cols)

hr = rawHR[all_cols]

# test
hr[hr['hhid'] == '       1   4']




,hhid,hml1,hml1a,hml2,hmlidx_1,hmlidx_2,hmlidx_3,hmlidx_4,hmlidx_5,hmlidx_6,hmlidx_7,hml3_1,hml3_2,hml3_3,hml3_4,hml3_5,hml3_6,hml3_7,hml4_1,hml4_2,hml4_3,hml4_4,hml4_5,hml4_6,hml4_7,hml5_1,hml5_2,hml5_3,hml5_4,hml5_5,hml5_6,hml5_7,hml6_1,hml6_2,hml6_3,hml6_4,hml6_5,hml6_6,hml6_7,hml7_1,...,hml39_13,hml39_14,hml39_15,hml39_16,hml39_17,hml39_18,hml39_19,hml39_20,hml39_21,hml39_22,hml39_23,hml39_24,hml39_25,hml39_26,hml40_01,hml40_02,hml40_03,hml40_04,hml40_05,hml40_06,hml40_07,hml40_08,hml40_09,hml40_10,hml40_11,hml40_12,hml40_13,hml40_14,hml40_15,hml40_16,hml40_17,hml40_18,hml40_19,hml40_20,hml40_21,hml40_22,hml40_23,hml40_24,hml40_25,hml40_26
3,1 4,3,3,0,1.0,2.0,3.0,NaN,NaN,NaN,NaN,"yes, seen","yes, seen","yes, seen",NaN,NaN,NaN,NaN,8.0,8.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,duranet plus,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
hr_cols = ['hhid']
hml10_ls = list(rawHR.filter(like='hml10'))
hr_cols.extend(hml10_ls)

#hml10 = rawHR.filter(like = 'hml10')
hr_df = rawHR[hr_cols].copy()
hr_df['total_ITNs'] = (hr_df[hml10_ls] == 'yes').sum(axis=1)
hr_df


total_itns_df = hr_df[['hhid', 'total_ITNs']]
total_itns_df

hmr_df = hmr_df.merge(total_itns_df, how='left', on='hhid')
hmr_df



,hhid,hv001,hvidx,hv005,hv024,hv103,hmlidx,hml10,hml12,hml1,hml1a,slept_ITN,use_numerator,itn_count,itn_capacity,hh_size,access_ratio,weighted_ratio,total_ITNs_x,total_ITNs_y
0,1 1,1,1,416820,kampala,yes,2.0,"no, don't know",only untreated nets,2,2,False,0,0.0,0.0,6,0.000000,0.0,0,0
1,1 1,1,2,416820,kampala,yes,2.0,"no, don't know",only untreated nets,2,2,False,0,0.0,0.0,6,0.000000,0.0,0,0
2,1 1,1,3,416820,kampala,yes,1.0,"no, don't know",only untreated nets,2,2,False,0,0.0,0.0,6,0.000000,0.0,0,0
3,1 1,1,4,416820,kampala,yes,1.0,"no, don't know",only untreated nets,2,2,False,0,0.0,0.0,6,0.000000,0.0,0,0
4,1 1,1,5,416820,kampala,yes,1.0,"no, don't know",only untreated nets,2,2,False,0,0.0,0.0,6,0.000000,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47958,342 30,342,2,0,refugee settlements,yes,2.0,yes,only treated (itn) nets,2,2,True,0,1.0,2.0,6,0.333333,0.0,2,2
47959,342 30,342,3,0,refugee settlements,yes,NaN,NaN,did not sleep under a net,2,2,False,0,1.0,2.0,6,0.333333,0.0,2,2
47960,342 30,342,4,0,refugee settlements,yes,NaN,NaN,did not sleep under a net,2,2,False,0,1.0,2.0,6,0.333333,0.0,2,2
47961,342 30,342,5,0,refugee settlements,yes,2.0,yes,only treated (itn) nets,2,2,True,0,1.0,2.0,6,0.333333,0.0,2,2


In [248]:
hmr_df['itn_capacity_2'] = hmr_df['total_ITNs_x'] * 2

# Compute ratio
hmr_df['access_ratio_2'] = np.minimum(hmr_df['itn_capacity_2'], hmr_df['hh_size']) / hmr_df['hh_size']

hmr_df

# calculate access
hmr_df['weighted_ratio_2'] = hmr_df.hv005 * hmr_df.access_ratio

# Group by region
region_df_access = hmr_df[['hv024', 'hv005', 'weighted_ratio_2']].groupby('hv024').sum().reset_index()

# Drop refugee settlements
region_df_access = region_df_access[region_df_access['hv005'] != 0]

# Divide
region_df_access['access_rate_2'] = region_df_access['weighted_ratio_2']/region_df_access['hv005']

region_df_access

<positron-console-cell-248>:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


,hv024,hv005,weighted_ratio_2,access_rate_2
0,kampala,1014956700,5.727107e+08,0.564271
1,south buganda,7169112974,4.449315e+09,0.620623
2,north buganda,5120003785,3.370096e+09,0.658221
3,busoga,3797009509,2.394948e+09,0.630746
4,bukedi,2585607226,1.609412e+09,0.622450
5,elgon,2059554498,1.366622e+09,0.663552
6,teso,2367811911,1.832715e+09,0.774012
7,lango,2437319984,1.858230e+09,0.762407
8,acholi,1953488373,1.120168e+09,0.573419
9,karamoja,1424291664,6.527313e+08,0.458285
